# RAG from Scratch: Cleaned LangChain Notebook

This version cleans up the original notebook so it works with current LangChain package structure and avoids common notebook pitfalls.

Key fixes:
- Uses `langchain_text_splitters` instead of deprecated/removed `langchain.text_splitter` imports.
- Removes `langchain hub` dependency and defines the prompt locally.
- Fixes typos: `langcahin_community`, `vectorestore`, and cosine similarity variable names.
- Avoids hardcoding API keys in the notebook.
- Uses `retriever.invoke(...)` instead of older retriever methods.
- Formats retrieved documents before sending them to the model.


## 0. Install packages

Run this once in your environment. After installation, restart the Jupyter kernel before continuing.


In [ ]:
%pip install -U langchain langchain-community langchain-core langchain-openai langchain-text-splitters chromadb beautifulsoup4 tiktoken numpy

## 1. Environment setup

Do **not** paste API keys directly into notebooks. Set them in your terminal or with `getpass` during the session.

Example terminal command on Windows PowerShell:

```powershell
setx OPENAI_API_KEY "your_key_here"
```

Then restart your terminal/Jupyter session.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")

# Optional LangSmith tracing. Leave this off unless you actively use LangSmith.
os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")


## 2. Imports

In [ ]:
import bs4
import numpy as np
import tiktoken

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


## 3. Tiny embedding example

In [ ]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

def num_tokens_from_string(text: str, encoding_name: str = "cl100k_base") -> int:
    """Return the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(text))

print("Question token count:", num_tokens_from_string(question))


In [ ]:
embeddings = OpenAIEmbeddings()

query_vector = embeddings.embed_query(question)
document_vector = embeddings.embed_query(document)

print("Embedding dimension:", len(query_vector))


In [ ]:
def cosine_similarity(vec1, vec2) -> float:
    """Compute cosine similarity between two vectors."""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_vector, document_vector)
print("Cosine similarity:", similarity)


## 4. Load documents

This loads Lilian Weng's agent blog post and only parses the main post content, title, and header.


In [ ]:
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={
        "parse_only": bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    },
)

docs = loader.load()

print(f"Loaded {len(docs)} document(s).")
print(docs[0].page_content[:500])


## 5. Split documents into chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)

splits = text_splitter.split_documents(docs)

print(f"Created {len(splits)} chunks.")
print(splits[0].page_content[:500])


## 6. Create vector store and retriever

In [ ]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [ ]:
retrieved_docs = retriever.invoke("What is Task Decomposition?")

print(f"Retrieved {len(retrieved_docs)} chunks.")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content[:700])


## 7. Build a basic generation chain

In [ ]:
def format_docs(docs):
    """Convert retrieved LangChain documents into one context string."""
    return "\n\n".join(doc.page_content for doc in docs)

template = """You are an assistant for question-answering tasks.

Answer the question using only the context below. If the answer is not in the context, say you do not know.
Keep the answer concise.

Question:
{question}

Context:
{context}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


In [ ]:
chain = prompt | llm | StrOutputParser()

answer = chain.invoke(
    {
        "context": format_docs(retrieved_docs),
        "question": "What is Task Decomposition?",
    }
)

print(answer)


## 8. Full RAG chain

This combines retrieval, formatting, prompting, generation, and output parsing into one runnable chain.


In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")


## 9. Try your own questions

In [ ]:
questions = [
    "What is Task Decomposition?",
    "What are some challenges with long-term planning in agents?",
    "What is the role of memory in LLM-powered agents?",
]

for q in questions:
    print(f"\nQuestion: {q}")
    print("Answer:", rag_chain.invoke(q))
